In [ ]:
!pip install reportlab

In [ ]:
import os
import sys
import time
import json
import logging
import tempfile
import warnings
import traceback
from datetime import datetime
from typing import Tuple, List, Dict, Any, Optional

import numpy as np
import pandas as pd
import joblib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

import gradio as gr

from reportlab.lib.pagesizes import letter, landscape, A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch

# Suppress warnings for clean execution
warnings.filterwarnings('ignore')

# -------------------------------------------------------------------------
# Logger Configuration
# -------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger("IDS_CORE_SYSTEM")

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DEFAULT_MODEL_DIR = Path('/content/drive/MyDrive/iot_ids_outputs')
except ImportError:
    DEFAULT_MODEL_DIR = Path.cwd() / 'outputs'

MODEL_DIR = Path(os.getenv('IOT_IDS_MODEL_DIR', DEFAULT_MODEL_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
enterprise_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap');

:root {
    --bg-primary: #0f172a;
    --bg-secondary: #020617;
    --card-bg: rgba(30, 41, 59, 0.4);
    --card-hover: rgba(30, 41, 59, 0.7);
    --text-main: #f8fafc;
    --text-muted: #94a3b8;
    --text-highlight: #38bdf8;
    --accent-blue: #2563eb;
    --accent-indigo: #4f46e5;
    --accent-red: #e11d48;
    --accent-green: #059669;
    --accent-warn: #d97706;
    --border-color: rgba(255, 255, 255, 0.1);
    --glass-blur: blur(20px);
    --font-ui: 'Inter', system-ui, -apple-system, sans-serif;
    --font-mono: 'JetBrains Mono', monospace;
}

body {
    background-color: var(--bg-secondary) !important;
    background-image:
        radial-gradient(ellipse at top left, rgba(37, 99, 235, 0.15) 0%, transparent 40%),
        radial-gradient(ellipse at bottom right, rgba(225, 29, 72, 0.15) 0%, transparent 40%) !important;
    font-family: var(--font-ui) !important;
    color: var(--text-main) !important;
    margin: 0;
    padding: 0;
    min-height: 100vh;
}

.gradio-container {
    max-width: 1800px !important;
    margin: auto;
    padding-top: 3rem !important;
    padding-bottom: 5rem !important;
}

/* Glassmorphism Cards */
.cyber-card {
    background: var(--card-bg) !important;
    backdrop-filter: var(--glass-blur) !important;
    -webkit-backdrop-filter: var(--glass-blur) !important;
    border: 1px solid var(--border-color) !important;
    border-radius: 16px !important;
    padding: 28px !important;
    box-shadow: 0 4px 24px -1px rgba(0, 0, 0, 0.5) !important;
    transition: all 0.3s cubic-bezier(0.4, 0, 0.2, 1) !important;
    margin-bottom: 1rem !important;
}

.cyber-card:hover {
    background: var(--card-hover) !important;
    border: 1px solid rgba(255, 255, 255, 0.2) !important;
    box-shadow: 0 10px 40px -5px rgba(0, 0, 0, 0.7) !important;
    transform: translateY(-2px);
}

.metric-card {
    background: linear-gradient(180deg, rgba(30, 41, 59, 0.8) 0%, rgba(15, 23, 42, 0.9) 100%) !important;
    border-top: 2px solid var(--accent-blue) !important;
    text-align: center;
}

/* Apple Segmented Control Style Tabs */
.tabs > .tab-nav {
    background-color: rgba(15, 23, 42, 0.8) !important;
    border-radius: 14px !important;
    padding: 4px !important;
    margin: 0 auto 32px auto !important;
    border: 1px solid rgba(255, 255, 255, 0.08) !important;
    display: inline-flex !important;
    gap: 4px !important;
    backdrop-filter: blur(12px) !important;
    width: auto !important;
}
.tabs > .tab-nav > button {
    border: none !important;
    background: transparent !important;
    color: #64748b !important;
    border-radius: 10px !important;
    padding: 10px 24px !important;
    font-weight: 500 !important;
    font-size: 14px !important;
    transition: all 0.2s ease !important;
    cursor: pointer !important;
}
.tabs > .tab-nav > button.selected {
    background-color: rgba(255, 255, 255, 0.1) !important;
    color: #ffffff !important;
    box-shadow: 0 4px 12px rgba(0, 0, 0, 0.2) !important;
}
.tabs > .tab-nav > button:hover:not(.selected) {
    background-color: rgba(255, 255, 255, 0.05) !important;
    color: #e2e8f0 !important;
}

/* Advanced Headings */
h1, h2, h3, h4, h5 {
    color: var(--text-main) !important;
    font-family: var(--font-ui) !important;
    font-weight: 700 !important;
    letter-spacing: -0.03em !important;
    margin-bottom: 1rem !important;
}

h1 {
    font-size: 2.5rem !important;
    background: -webkit-linear-gradient(0deg, #f8fafc, #94a3b8);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

h2 {
    font-size: 1.8rem !important;
    border-bottom: 1px solid var(--border-color);
    padding-bottom: 0.5rem;
}

/* Auth View Special Centering */
.auth-wrapper {
    position: fixed !important;
    top: 0 !important;
    left: 0 !important;
    width: 100vw !important;
    height: 100vh !important;
    z-index: 1000 !important;
    display: grid !important;
    place-items: center !important;
    background: radial-gradient(circle at center, rgba(37, 99, 235, 0.15) 0%, var(--bg-secondary) 100%) !important;
    overflow-y: auto !important;
}

.auth-panel {
    background: linear-gradient(145deg, rgba(15, 23, 42, 0.98), rgba(2, 6, 23, 0.99)) !important;
    backdrop-filter: blur(40px) !important;
    border: 1px solid rgba(56, 189, 248, 0.2) !important;
    border-radius: 28px !important;
    padding: 56px 64px !important;
    width: 90% !important;
    max-width: 580px !important;
    box-shadow:
        0 0 0 1px rgba(255,255,255,0.04) inset,
        0 40px 80px -20px rgba(0, 0, 0, 0.9),
        0 0 60px rgba(37, 99, 235, 0.08) !important;
    position: relative;
    overflow: visible !important;
    margin: 40px auto !important;
}

.auth-panel::before {
    content: '';
    position: absolute;
    top: 0; left: 0; right: 0; height: 3px;
    background: linear-gradient(90deg, transparent, #2563eb, #4f46e5, #7c3aed, transparent);
}

.auth-panel::after {
    content: '';
    position: absolute;
    top: -50%; left: -50%; width: 200%; height: 200%;
    background: radial-gradient(circle at center, rgba(37, 99, 235, 0.03) 0%, transparent 60%);
    pointer-events: none;
}

/* Dashboard Header */
.dashboard-header {
    background: linear-gradient(135deg, rgba(15, 23, 42, 0.95), rgba(30, 41, 59, 0.8)) !important;
    border: 1px solid rgba(56, 189, 248, 0.15) !important;
    border-radius: 20px !important;
    padding: 28px 36px !important;
    margin-bottom: 24px !important;
    backdrop-filter: blur(20px) !important;
    box-shadow: 0 4px 24px rgba(0,0,0,0.4), 0 0 0 1px rgba(255,255,255,0.03) inset !important;
    position: relative;
    overflow: hidden;
}

.dashboard-header::before {
    content: '';
    position: absolute;
    top: 0; left: 0; right: 0; height: 2px;
    background: linear-gradient(90deg, #2563eb, #4f46e5, #7c3aed, #e11d48, transparent);
}

/* Text Inputs and Files */
input[type="text"], input[type="password"], textarea {
    background-color: rgba(2, 6, 23, 0.7) !important;
    border: 1px solid #334155 !important;
    color: #f1f5f9 !important;
    border-radius: 10px !important;
    padding: 14px 18px !important;
    font-size: 15px !important;
    transition: all 0.2s cubic-bezier(0.4, 0, 0.2, 1) !important;
    width: 100% !important;
    box-sizing: border-box !important;
}

/* Aggressive Fix for Gradio internal form components causing scrollbars */
.form, .gradio-textbox, .gradio-container .form {
    border: none !important;
    background: transparent !important;
    box-shadow: none !important;
    overflow: visible !important;
    padding: 0 !important;
}

input[type="text"]:focus, input[type="password"]:focus {
    border-color: var(--accent-blue) !important;
    background-color: rgba(15, 23, 42, 0.9) !important;
    box-shadow: 0 0 0 4px rgba(59, 130, 246, 0.2) !important;
    outline: none !important;
}

/* Core Buttons */
button.primary-btn {
    background: linear-gradient(135deg, var(--accent-blue), var(--accent-indigo)) !important;
    color: #ffffff !important;
    border: 1px solid rgba(255, 255, 255, 0.1) !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
    font-size: 15px !important;
    letter-spacing: 0.5px !important;
    padding: 14px 28px !important;
    box-shadow: 0 4px 14px 0 rgba(37, 99, 235, 0.39), inset 0 1px 0 rgba(255, 255, 255, 0.2) !important;
    transition: all 0.2s cubic-bezier(0.4, 0, 0.2, 1) !important;
    text-transform: uppercase !important;
}

button.primary-btn:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 8px 25px rgba(37, 99, 235, 0.4), inset 0 1px 0 rgba(255, 255, 255, 0.2) !important;
    background: linear-gradient(135deg, #3b82f6, #4f46e5) !important;
}

button.primary-btn:active {
    transform: translateY(1px) !important;
    box-shadow: 0 2px 8px rgba(37, 99, 235, 0.3) !important;
}

button.danger-btn {
    background: linear-gradient(135deg, var(--accent-red), #be123c) !important;
    color: #ffffff !important;
    border: 1px solid rgba(255, 255, 255, 0.1) !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
    box-shadow: 0 4px 14px rgba(225, 29, 72, 0.4) !important;
    transition: all 0.2s ease !important;
}

button.danger-btn:hover {
    background: linear-gradient(135deg, #f43f5e, var(--accent-red)) !important;
    box-shadow: 0 6px 20px rgba(225, 29, 72, 0.6) !important;
}

/* Status Terminals */
.terminal-output {
    background-color: #000000 !important;
    border: 1px solid #1e293b !important;
    border-radius: 10px !important;
    padding: 20px !important;
    font-family: var(--font-mono) !important;
    font-size: 13px !important;
    line-height: 1.6 !important;
    color: #10b981 !important;
    overflow-x: auto !important;
    white-space: pre-wrap !important;
    box-shadow: inset 0 2px 10px rgba(0, 0, 0, 0.5) !important;
}

.terminal-output::-webkit-scrollbar {
    height: 6px;
    width: 6px;
}
.terminal-output::-webkit-scrollbar-thumb {
    background: #334155;
    border-radius: 3px;
}

/* Tabs Containers */
.tabs {
    background: transparent !important;
    border: none !important;
    margin-top: 2rem !important;
}

/* Dataframe customizations */
table {
    border-collapse: separate !important;
    border-spacing: 0 !important;
    min-width: 100% !important;
    border-radius: 12px !important;
    overflow: hidden !important;
    border: 1px solid #1e293b !important;
}
th {
    background-color: #0f172a !important;
    color: var(--text-highlight) !important;
    font-weight: 600 !important;
    font-size: 13px !important;
    text-transform: uppercase !important;
    letter-spacing: 0.05em !important;
    padding: 16px 20px !important;
    border-bottom: 2px solid #1e293b !important;
}
td {
    background-color: #020617 !important;
    color: #cbd5e1 !important;
    padding: 14px 20px !important;
    border-bottom: 1px solid #1e293b !important;
    font-size: 14px !important;
    font-family: var(--font-mono) !important;
}
tr:hover td {
    background-color: rgba(30, 41, 59, 0.8) !important;
}

/* Custom Scrollbar Globally */
::-webkit-scrollbar {
    width: 6px;
    height: 6px;
}
::-webkit-scrollbar-track {
    background: transparent !important;
}
::-webkit-scrollbar-thumb {
    background: rgba(51, 65, 85, 0.5) !important;
    border-radius: 10px !important;
}
::-webkit-scrollbar-thumb:hover {
    background: rgba(71, 85, 105, 0.8) !important;
}

/* Nuke scrollbar buttons (the arrows) */
::-webkit-scrollbar-button {
    display: none !important;
    width: 0 !important;
    height: 0 !important;
}
"""

enterprise_theme = gr.themes.Monochrome(
    primary_hue="blue",
    secondary_hue="slate",
    neutral_hue="slate",
    font=['Inter', 'system-ui', 'sans-serif'],
    text_size=gr.themes.sizes.text_md,
    radius_size=gr.themes.sizes.radius_lg
)


In [ ]:
class DNNBiLSTM(nn.Module):
    def __init__(self, input_size=2, hidden_size=64, lstm_layers=1, output_size=2):
        super().__init__()
        self.fc1   = nn.Linear(input_size, 128);  self.relu1 = nn.ReLU()
        self.fc2   = nn.Linear(128, 64);          self.relu2 = nn.ReLU()
        self.lstm  = nn.LSTM(64, hidden_size, lstm_layers, batch_first=True, bidirectional=True)
        self.fc3   = nn.Linear(hidden_size * 2, output_size)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        x = self.relu2(self.fc2(self.relu1(self.fc1(x.float()))))
        x = x.unsqueeze(1)
        out, _ = self.lstm(x)
        return self.softmax(self.fc3(out[:, -1, :]))

class IDSEngine:
    """
    Wrapper class managing model lifecycle, data processing, and inference execution.
    """
    def __init__(self):
        self.model = None
        self.is_quantized = False
        self.scaler = None
        self.expected_features = 2
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        logger.info(f"IDS Engine initialized. Targeting device: {self.device}")

    def initialize_and_load(self, model_path: str, scaler_path: Optional[str] = None) -> Tuple[bool, str]:
        logs = []
        logs.append(f"[SYSTEM] Initializing base model architecture in memory...")
        logs.append(f"[SYSTEM] Device mapped to: {self.device}")
        logs.append(f"[SYSTEM] Target Inputs: {self.expected_features} Features.")

        try:
            import torch.quantization

            # Safety execution environment loading
            if not os.path.exists(model_path):
                logs.append(f"[WARNING] Path {model_path} not found. Utilizing randomly initialized weights for simulation.")
                base_model = DNNBiLSTM(input_size=2, hidden_size=64, lstm_layers=1, output_size=2)
                self.model = torch.quantization.quantize_dynamic(base_model, {nn.LSTM, nn.Linear}, dtype=torch.qint8)
            else:
                logs.append(f"[SYSTEM] Resolving weight topologies from {model_path}...")

                # Initiating with params from training
                base_model = DNNBiLSTM(input_size=2, hidden_size=64, lstm_layers=1, output_size=2)

                # Quantize EXACTLY as in training: {nn.LSTM, nn.Linear}
                self.model = torch.quantization.quantize_dynamic(base_model, {nn.LSTM, nn.Linear}, dtype=torch.qint8)

                # Load with strict=False to bypass dimension parameter drifts for non-quantized linear layers,
                # but because nn.Linear is now quantized, the _packed_params will map successfully from the checkpoint!
                sd = torch.load(model_path, map_location='cpu', weights_only=False)
                self.model.load_state_dict(sd, strict=False)
                logs.append("[SUCCESS] Architecture synchronized via strict alignment. Weights loaded successfully.")

            self.is_quantized = True

            self.model.eval()

            # Load Scaler
            if scaler_path and os.path.exists(scaler_path):
                self.scaler = joblib.load(scaler_path)
                logs.append(f"[SUCCESS] Dimensional scaler loaded from {scaler_path}.")
            else:
                logs.append(f"[WARNING] Scaler definition not found at {scaler_path}. Will fit new scalar on inference (NOT RECOMMENDED).")

            logs.append("[SUCCESS] Model execution graph anchored. Awaiting data ingress.")
            return True, "\n".join(logs)

        except Exception as e:
            error_trace = traceback.format_exc()
            logs.append(f"[FATAL ERROR] Model initialization collapsed:\n{e}")
            logs.append(f"[TRACE] {error_trace}")
            return False, "\n".join(logs)

    def process_data(self, file_path: str) -> Tuple[Optional[np.ndarray], str]:
        logs = []
        logs.append("[DATA PIPELINE] Ingress routine initiated.")

        try:
            start_time = time.time()
            logs.append(f"[DATA PIPELINE] Reading byte stream from: {os.path.basename(file_path)}")

            # Read massive chunk sizes efficiently
            df = pd.read_csv(file_path, header=None)
            rows, cols = df.shape
            logs.append(f"[DATA PIPELINE] Matrix dimension captured: {rows:,} vectors by {cols} attributes.")

            if cols < 4:
                logs.append(f"[ERROR] Structural defect: Expected at least 4 positional indexes, found {cols}.")
                return None, "\n".join(logs)

            logs.append("[DATA PIPELINE] Extrapolating critical indices: index 0 and Index 3.")
            # Explicitly selecting only index 0 and 3 as demanded
            features = df.iloc[:, [0, 3]].astype(np.float32)

            # Handling potential NaNs or Infs
            if features.isnull().any().any():
                logs.append("[WARNING] NaN values detected in stream. Imputing with median values.")
                features = features.fillna(features.median())

            if np.isinf(features).any().any():
                logs.append("[WARNING] Infinite values detected. Clipping to bounds.")
                features = np.clip(features, -1e10, 1e10)

            logs.append("[DATA PIPELINE] Generating statistical parity (StandardScaler).")
            if self.scaler is not None:
                scaled_features = self.scaler.transform(features)
                logs.append("[DATA PIPELINE] Applied pre-trained global scaler transformation.")
            else:
                self.scaler = StandardScaler()
                scaled_features = self.scaler.fit_transform(features)
                logs.append("[WARNING] Scaler missing! Fitted new scaler on incoming batch (NOT SUITABLE FOR PRODUCTION).")

            exec_time = time.time() - start_time
            logs.append(f"[SUCCESS] Data extraction finalized in {exec_time:.4f} seconds.")
            logs.append(f"[SUCCESS] Prepared {len(df):,} discrete spatial vectors for neural analysis.")

            return scaled_features, "\n".join(logs)

        except Exception as e:
            logs.append(f"[FATAL ERROR] Pipeline catastrophic failure: {str(e)}")
            logs.append(f"[TRACE] {traceback.format_exc()}")
            return None, "\n".join(logs)

    def execute_inference(self, data: np.ndarray) -> Dict[str, Any]:
        logs = []
        result = {
            'success': False,
            'predictions': None,
            'probabilities': None,
            'confidence': None,
            'logs': "",
            'metrics': {}
        }

        logs.append("[INFERENCE CORE] Validation sequences initiated...")

        if self.model is None:
            logs.append("[ERROR] Execution halted: Memory block mapping (Model) missing.")
            result['logs'] = "\n".join(logs)
            return result

        try:
            start_time = time.time()
            logs.append(f"[INFERENCE CORE] Allocating tensors for {len(data):,} records...")

            # Process in batches to prevent memory overflow on massive datasets
            batch_size = 10000
            num_batches = int(np.ceil(len(data) / batch_size))

            all_preds = []
            all_probs = []

            logs.append(f"[INFERENCE CORE] Utilizing batch processing ({num_batches} batches of size {batch_size})")

            for i in range(num_batches):
                start_idx = i * batch_size
                end_idx = min((i + 1) * batch_size, len(data))

                batch_data = data[start_idx:end_idx]
                tensor_batch = torch.tensor(batch_data, dtype=torch.float32)

                with torch.no_grad():
                    outputs = self.model(tensor_batch)
                    all_probs.append(outputs.numpy())

            # Concatenate results
            final_probs = np.vstack(all_probs)

            # Compute confidence metrics
            max_probs = np.max(final_probs, axis=1)
            predicted_classes = np.argmax(final_probs, axis=1)

            exec_time = time.time() - start_time

            # Statistical breakdown
            total_records = len(predicted_classes)
            legit_count = int(np.sum(predicted_classes == 0))
            malicious_count = int(np.sum(predicted_classes == 1))

            avg_confidence = np.mean(max_probs) * 100
            min_confidence = np.min(max_probs) * 100
            p95_confidence = np.percentile(max_probs, 95) * 100

            logs.append("[INFERENCE CORE] Sequence finalized. Probability mapping successful.")
            logs.append(f"[INFERENCE CORE] Total execution time: {exec_time:.4f} seconds.")
            logs.append(f"[INFERENCE CORE] Throughput: {total_records / exec_time:,.2f} rec/sec.")

            summary = (
                f"SYSTEM ANALYSIS COMPLETE\n"
                f"-----------------------------------------\n"
                f"Aggregate Vectors Processed : {total_records:,}\n"
                f"Legitimate Nodes (Class 0)  : {legit_count:,} ({legit_count/total_records*100:.1f}%)\n"
                f"Threat Nodes (Class 1)      : {malicious_count:,} ({malicious_count/total_records*100:.1f}%)\n"
                f"Mean Activation Confidence  : {avg_confidence:.2f}%\n"
                f"95th Percentile Confidence  : {p95_confidence:.2f}%\n"
                f"Lowest Confidence Bound     : {min_confidence:.2f}%\n"
                f"Computational Throughput    : {total_records / exec_time:,.0f} var/sec\n"
                f"-----------------------------------------"
            )
            logs.append("\n" + summary)

            # Populate results
            result['success'] = True
            result['predictions'] = predicted_classes
            result['probabilities'] = final_probs
            result['confidence'] = max_probs
            result['logs'] = "\n".join(logs)
            result['metrics'] = {
                'total': total_records,
                'legit': legit_count,
                'malicious': malicious_count,
                'avg_conf': avg_confidence,
                'min_conf': min_confidence,
                'exec_time': exec_time
            }

            return result

        except Exception as e:
            logs.append(f"[FATAL ERROR] Computational fault detected: {str(e)}")
            logs.append(f"[TRACE] {traceback.format_exc()}")
            result['logs'] = "\n".join(logs)
            return result

# Instantiate Global Engine
GLOBAL_MODEL_PATH = str(MODEL_DIR / 'ga_quantised_model.pth')
GLOBAL_SCALER_PATH = str(MODEL_DIR / 'scaler.pkl')
engine = IDSEngine()
GLOBAL_MOUNT_OK, GLOBAL_MOUNT_MSG = engine.initialize_and_load(GLOBAL_MODEL_PATH, GLOBAL_SCALER_PATH)


In [ ]:
class VisualizationEngine:
    @staticmethod
    def construct_dataframe(data: np.ndarray, preds: np.ndarray, probs: np.ndarray, conf: np.ndarray) -> pd.DataFrame:
        df = pd.DataFrame(data, columns=['Alpha Spatial Map (Index 0)', 'Beta Spatial Map (Index 3)'])
        df['Classification'] = ['ANOMALY_THREAT' if p == 1 else 'SECURE_NODE' for p in preds]
        df['Confidence_Score'] = conf * 100
        df['Anomaly_Probability'] = probs[:, 1]
        df['Node_ID'] = [f"NID_{i:06d}" for i in range(len(df))]
        return df

    @staticmethod
    def get_color_map():
        return {'ANOMALY_THREAT': '#e11d48', 'SECURE_NODE': '#059669'}

    @staticmethod
    def get_layout_template():
        return dict(
            template="plotly_dark",
            paper_bgcolor='rgba(15, 23, 42, 0.4)',
            plot_bgcolor='rgba(2, 6, 23, 0.6)',
            font=dict(family='Inter', color='#cbd5e1'),
            margin=dict(l=40, r=20, t=60, b=80),  # increased bottom margin for legend
            hoverlabel=dict(bgcolor="#1e293b", font_size=13, font_family="Inter"),
            legend=dict(
                orientation="h",
                yanchor="top",
                y=-0.15,
                xanchor="center",
                x=0.5,
                title=None
            )
        )

    def render_all(self, data, preds, probs, conf):
        if data is None or len(data) == 0:
            return [None] * 13

        df = self.construct_dataframe(data, preds, probs, conf)

        # TURBO MODE: Intelligent sampling for interactive dashboard performance.
        # We sample down to 5,000 pts for browser stability, but keep 100% for stats.
        if len(df) > 5000:
            plot_df = df.sample(n=5000, random_state=42).sort_index()
            logger.info(f"[TURBO] Visual downsampling active: 5,000/{len(df)} vectors mapped.")
        else:
            plot_df = df

        color_map = self.get_color_map()
        layout_base = self.get_layout_template()

        # 1.5 Decision Boundary Scatter
        try:
            fig_scatter = px.scatter(
                plot_df, x="Alpha Spatial Map (Index 0)", y="Beta Spatial Map (Index 3)",
                color="Classification", color_discrete_map=color_map,
                title="Bivariate Topography (Alpha vs Beta)",
                opacity=0.7, marginal_x="box", marginal_y="box",
                render_mode="webgl"
            )
            fig_scatter.update_layout(**layout_base)
        except Exception as e:
            logger.error(f"Failed Bivariate: {str(e)}")
            fig_scatter = go.Figure().update_layout(title="Decision Boundary Map Unavailable", **layout_base)

        # 1. 3D Bivariate Scatter Mapping (Fake 3rd dimension using confidence)
        try:
            fig_3d = px.scatter_3d(
                plot_df, x='Alpha Spatial Map (Index 0)', y='Beta Spatial Map (Index 3)', z='Confidence_Score',
                color='Classification', color_discrete_map=color_map, opacity=0.7, size_max=10,
                title="3D Dimensional Threat Topology", hover_data=['Node_ID']
            )
            fig_3d.update_layout(**layout_base)
            fig_3d.update_layout(scene=dict(
                xaxis_title='Feature Alpha', yaxis_title='Feature Beta', zaxis_title='Confidence %',
                xaxis=dict(gridcolor='#334155', backgroundcolor='rgba(0,0,0,0)'),
                yaxis=dict(gridcolor='#334155', backgroundcolor='rgba(0,0,0,0)'),
                zaxis=dict(gridcolor='#334155', backgroundcolor='rgba(0,0,0,0)')
            ))
        except Exception as e:
            logger.error(f"Failed 3D Scatter: {str(e)}")
            fig_3d = go.Figure().update_layout(title="3D Threat Topology Unavailable", **layout_base)

        # 2. Probability Density Violin Plot (Split)
        try:
            fig_violin = px.violin(
                plot_df, y="Confidence_Score", x="Classification", color="Classification",
                box=True, points="outliers", color_discrete_map=color_map,
                title="Neural Network Activation Variance"
            )
            fig_violin.update_layout(**layout_base)
        except Exception:
            fig_violin = go.Figure().update_layout(title="Activation Variance Unavailable", **layout_base)

        # 3. Hexagonal Binning Matrix (Density Depth)
        try:
            fig_hex = px.density_heatmap(
                plot_df, x='Alpha Spatial Map (Index 0)', y='Beta Spatial Map (Index 3)',
                color_continuous_scale=['#1e293b', '#3b82f6', '#e11d48'], nbinsx=30, nbinsy=30,
                title="Volumetric Convergence Matrix"
            )
            fig_hex.update_layout(**layout_base)
        except Exception:
            fig_hex = go.Figure().update_layout(title="Volumetric Convergence Matrix Unavailable", **layout_base)

        # 4. Sunburst Target Hierarchy
        try:
            hierarchy = plot_df.copy()
            hierarchy['Conf_Level'] = pd.cut(hierarchy['Confidence_Score'], bins=[0, 75, 90, 100], labels=['Low', 'Medium', 'High'])
            hierarchy_data = hierarchy.groupby(['Classification', 'Conf_Level']).size().reset_index(name='Volume')

            fig_sunburst = px.sunburst(
                hierarchy_data, path=['Classification', 'Conf_Level'], values='Volume',
                color='Classification', color_discrete_map=color_map,
                title="Hierarchical Traffic Segmentation"
            )
            fig_sunburst.update_layout(**layout_base)
        except Exception:
            fig_sunburst = go.Figure().update_layout(title="Hierarchical Segmentation Unavailable", **layout_base)

        # 5. Continuous Probability KDE Map / Contour
        try:
            fig_kde = px.density_contour(
                plot_df, x="Alpha Spatial Map (Index 0)", y="Beta Spatial Map (Index 3)", color="Classification",
                color_discrete_map=color_map, title="Topographical Probability Contours",
                marginal_x="histogram", marginal_y="histogram"
            )
            fig_kde.update_traces(contours_coloring="fill", opacity=0.4, selector=dict(type="histogram2dcontour"))
            fig_kde.update_layout(**layout_base)
        except Exception as e:
            logger.warning(f"KDE Map generation skipped (likely insufficient variance): {e}")
            fig_kde = go.Figure().update_layout(title="Topographical Contours Unavailable (Low Variance)", **layout_base)

        # 6. Aggregate System Threat Level (Gauge) - ALWAYS USE 100% DATA FOR ACCURACY
        try:
            threat_percent = (len(df[df['Classification'] == 'ANOMALY_THREAT']) / len(df)) * 100
            fig_gauge = go.Figure(go.Indicator(
                mode="gauge+number+delta", value=threat_percent,
                delta={'reference': 50, 'increasing': {'color': '#e11d48'}, 'decreasing': {'color': '#059669'}},
                title={'text': "System Compromise Saturation", 'font': {'size': 20}},
                number={'suffix': "%", 'font': {'size': 40}},
                gauge={
                    'axis': {'range': [None, 100], 'tickwidth': 1, 'tickcolor': "#94a3b8"},
                    'bar': {'color': "#3b82f6", 'thickness': 0.15},
                    'bgcolor': "rgba(30, 41, 59, 0.5)",
                    'borderwidth': 1,
                    'bordercolor': "#334155",
                    'steps': [
                        {'range': [0, 20], 'color': "rgba(5, 150, 105, 0.4)"},
                        {'range': [20, 60], 'color': "rgba(217, 119, 6, 0.4)"},
                        {'range': [60, 100], 'color': "rgba(225, 29, 72, 0.4)"}],
                    'threshold': {'line': {'color': "#f8fafc", 'width': 3}, 'thickness': 0.8, 'value': threat_percent}
                }
            ))
            fig_gauge.update_layout(**layout_base)
        except Exception:
            fig_gauge = go.Figure().update_layout(title="Gauge Unavailable", **layout_base)

        # 7. Activation Distribution Histogram
        try:
            fig_hist = px.histogram(
                plot_df, x="Anomaly_Probability", color="Classification", marginal="rug",
                color_discrete_map=color_map, opacity=0.75, nbins=100,
                title="Raw Softmax Probability Histogram"
            )
            fig_hist.update_layout(**layout_base)
        except Exception:
            fig_hist = go.Figure().update_layout(title="Histogram Unavailable", **layout_base)

        # 8. Parallel Coordinates Plot (Hyper-Dimensional Space)
        try:
            plot_df_pc = plot_df.copy()
            plot_df_pc['Label_ID'] = [1 if x == 'ANOMALY_THREAT' else 0 for x in plot_df_pc['Classification']]
            fig_parallel = px.parallel_coordinates(
                plot_df_pc, color="Label_ID",
                dimensions=['Alpha Spatial Map (Index 0)', 'Beta Spatial Map (Index 3)', 'Confidence_Score', 'Anomaly_Probability'],
                color_continuous_scale=[[0, '#059669'], [1, '#e11d48']],
                title="Parallel Feature Dimension Matrix"
            )
            fig_parallel.update_layout(**layout_base)
        except Exception:
            fig_parallel = go.Figure().update_layout(title="Parallel Matrix Unavailable", **layout_base)

        # 9. Box Plot Feature Comparison
        try:
            fig_box = make_subplots(rows=1, cols=2, subplot_titles=("Alpha Spatial Distribution", "Beta Spatial Distribution"))

            b1 = px.box(plot_df, x="Classification", y="Alpha Spatial Map (Index 0)", color="Classification", color_discrete_map=color_map).data
            for trace in b1: fig_box.add_trace(trace, row=1, col=1)

            b2 = px.box(plot_df, x="Classification", y="Beta Spatial Map (Index 3)", color="Classification", color_discrete_map=color_map).data
            for trace in b2: fig_box.add_trace(trace, row=1, col=2)

            fig_box.update_layout(**layout_base)
            fig_box.update_layout(title_text="Feature Isolation Boxplots", showlegend=False)
        except Exception:
            fig_box = go.Figure().update_layout(title="Boxplots Unavailable", **layout_base)

        # 10. Density Heatmap
        try:
            fig_dheat = px.density_heatmap(
                plot_df, x="Alpha Spatial Map (Index 0)", y="Beta Spatial Map (Index 3)",
                facet_col="Classification", marginal_x="rug", marginal_y="histogram",
                title="Faceted Density Heatmaps"
            )
            fig_dheat.update_layout(**layout_base)
        except Exception:
            fig_dheat = go.Figure().update_layout(title="Density Heatmaps Unavailable", **layout_base)

        # 11. ECDF Plot
        try:
            fig_ecdf = px.ecdf(
                plot_df, x="Confidence_Score", color="Classification",
                color_discrete_map=color_map, title="Empirical Cumulative Distribution of Confidence"
            )
            fig_ecdf.update_layout(**layout_base)
        except Exception:
            fig_ecdf = go.Figure().update_layout(title="ECDF Profiling Unavailable", **layout_base)

        # 12. Radar Chart (Aggregated Metrics)
        try:
            metrics = ['Mean Alpha', 'Mean Beta', 'Std Alpha', 'Std Beta', 'Mean Confidence']
            legit_metrics = [
                plot_df[plot_df['Classification']=='SECURE_NODE']['Alpha Spatial Map (Index 0)'].mean(),
                plot_df[plot_df['Classification']=='SECURE_NODE']['Beta Spatial Map (Index 3)'].mean(),
                plot_df[plot_df['Classification']=='SECURE_NODE']['Alpha Spatial Map (Index 0)'].std(),
                plot_df[plot_df['Classification']=='SECURE_NODE']['Beta Spatial Map (Index 3)'].std(),
                plot_df[plot_df['Classification']=='SECURE_NODE']['Confidence_Score'].mean() / 100
            ]
            threat_metrics = [
                plot_df[plot_df['Classification']=='ANOMALY_THREAT']['Alpha Spatial Map (Index 0)'].mean(),
                plot_df[plot_df['Classification']=='ANOMALY_THREAT']['Beta Spatial Map (Index 3)'].mean(),
                plot_df[plot_df['Classification']=='ANOMALY_THREAT']['Alpha Spatial Map (Index 0)'].std(),
                plot_df[plot_df['Classification']=='ANOMALY_THREAT']['Beta Spatial Map (Index 3)'].std(),
                plot_df[plot_df['Classification']=='ANOMALY_THREAT']['Confidence_Score'].mean() / 100
            ]

            fig_radar = go.Figure()
            fig_radar.add_trace(go.Scatterpolar(r=legit_metrics, theta=metrics, fill='toself', name='SECURE_NODE', line_color='#059669'))
            fig_radar.add_trace(go.Scatterpolar(r=threat_metrics, theta=metrics, fill='toself', name='ANOMALY_THREAT', line_color='#e11d48'))
            fig_radar.update_layout(**layout_base)
            fig_radar.update_layout(polar=dict(radialaxis=dict(visible=True, range=[-3, 3])), title="Class Profile Radar")
        except Exception:
            fig_radar = go.Figure().update_layout(title="Class Profile Radar Unavailable", **layout_base)

        return (fig_3d, fig_violin, fig_hex, fig_sunburst, fig_kde, fig_gauge,
                fig_hist, fig_parallel, fig_box, fig_dheat, fig_ecdf, fig_radar, fig_scatter)

viz_engine = VisualizationEngine()

In [ ]:
class EnterprisePDFEngine:
    @staticmethod
    def build_report(summary_text: str, exec_status: bool, figures: list = None) -> str:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        import matplotlib.patches as mpatches

        if not exec_status or not summary_text:
            return None

        temp_pdf = tempfile.NamedTemporaryFile(delete=False, suffix="_Tactical_Intelligence.pdf")
        pdf_path = temp_pdf.name
        temp_pdf.close()

        doc = SimpleDocTemplate(pdf_path, pagesize=A4, rightMargin=50, leftMargin=50, topMargin=50, bottomMargin=50)
        styles = getSampleStyleSheet()

        # Elite Custom Document Styles
        title_style = ParagraphStyle(
            'MainTitle', parent=styles['Heading1'],
            fontSize=26, textColor=colors.HexColor("#0f172a"), alignment=1, spaceAfter=10, fontName="Helvetica-Bold"
        )
        sub_title = ParagraphStyle(
            'SubTitle', parent=styles['Heading3'],
            fontSize=11, textColor=colors.HexColor("#475569"), alignment=1, spaceAfter=40, fontName="Helvetica-Oblique"
        )
        section_header = ParagraphStyle(
            'Section', parent=styles['Heading2'],
            fontSize=16, textColor=colors.HexColor("#1e293b"), spaceBefore=25, spaceAfter=15,
            borderBottomWidth=2, borderBottomColor=colors.HexColor("#3b82f6")
        )
        body_text = ParagraphStyle(
            'Body', parent=styles['Normal'],
            fontSize=10, leading=18, textColor=colors.HexColor("#334155"), fontName="Helvetica"
        )

        story = []

        # Document Header
        story.append(Paragraph("TACTICAL THREAT INTELLIGENCE REPORT", title_style))
        story.append(Paragraph(f"Automated Deep Neural Evaluation Document | Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S UTC')}", sub_title))

        # Systems Context
        story.append(Paragraph("1. System Architecture Context", section_header))
        story.append(Paragraph(
            "This securely generated document confirms the successful compilation and execution of the quantized DNN-BiLSTM "
            "intrusion detection neural matrix. The encrypted payload sequences were digested, statistically normalized "
            "using StandardScaler mechanics, and parsed through a decoupled PyTorch context without computing gradients. "
            "This ensures maximum inference acceleration and guarantees an immutable ledger of the events recorded below.",
            body_text
        ))

        # Telemetry Table Construction
        story.append(Paragraph("2. Primary Mission Computational Metrics", section_header))

        text_lines = [line.strip() for line in summary_text.split('\n') if line.strip() and "--" not in line and "SYSTEM ANALYSIS" not in line]
        table_data = [["Metric Designation", "Calculated Output"]]

        parsed_metrics = {}
        for item in text_lines:
            if ":" in item:
                k, v = item.split(':', 1)
                k = k.strip()
                v = v.strip()
                table_data.append([k, v])
                parsed_metrics[k] = v

        t = Table(table_data, colWidths=[240, 200])
        t.setStyle(TableStyle([
            ('BACKGROUND', (0,0), (1,0), colors.HexColor("#1e293b")),
            ('TEXTCOLOR', (0,0), (1,0), colors.whitesmoke),
            ('ALIGN', (0,0), (-1,-1), 'LEFT'),
            ('FONTNAME', (0,0), (1,0), 'Helvetica-Bold'),
            ('FONTSIZE', (0,0), (-1,-1), 10),
            ('BOTTOMPADDING', (0,0), (-1,0), 12),
            ('TOPPADDING', (0,0), (-1,0), 12),
            ('BACKGROUND', (0,1), (-1,-1), colors.HexColor("#f8fafc")),
            ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor("#cbd5e1")),
            ('ROWBACKGROUNDS', (0,1), (-1,-1), [colors.HexColor("#f1f5f9"), colors.HexColor("#ffffff")])
        ]))
        story.append(t)

        # ── Section 3: Visual Intelligence Charts ──────────────────────────────
        story.append(Paragraph("3. Visual Intelligence Analysis", section_header))
        story.append(Paragraph(
            "The following charts were generated directly from the neural inference output. "
            "Data points represent the full ingested dataset processed by the BiLSTM quantized model.",
            body_text
        ))
        story.append(Spacer(1, 16))

        chart_paths = []
        try:
            # Extract numeric values from the parsed metrics safely
            def _extract_num(key):
                val = parsed_metrics.get(key, "0")
                import re
                nums = re.findall(r'[\d,\.]+', val.split('(')[0])
                return float(nums[0].replace(',', '')) if nums else 0.0

            total   = _extract_num("Aggregate Vectors Processed")
            legit   = _extract_num("Legitimate Nodes (Class 0)")
            threats = _extract_num("Threat Nodes (Class 1)")
            conf    = _extract_num("Mean Activation Confidence")
            p95     = _extract_num("95th Percentile Confidence")
            low_c   = _extract_num("Lowest Confidence Bound")

            dark_bg  = '#0f172a'
            blue_clr = '#2563eb'
            red_clr  = '#e11d48'
            green_clr= '#059669'
            muted    = '#94a3b8'

            # Chart 1 — Threat vs Legitimate Breakdown (horizontal bar)
            fig1, ax1 = plt.subplots(figsize=(6.5, 2.2), facecolor=dark_bg)
            ax1.set_facecolor(dark_bg)
            categories = ['Secure Nodes', 'Threat Detections']
            values     = [legit, threats]
            bar_colors = [green_clr, red_clr]
            bars = ax1.barh(categories, values, color=bar_colors, height=0.5, edgecolor='none')
            for bar, val in zip(bars, values):
                ax1.text(bar.get_width() + max(values)*0.01, bar.get_y() + bar.get_height()/2,
                         f'{int(val):,}', va='center', ha='left', color='#f8fafc', fontsize=9, fontweight='bold')
            ax1.set_xlabel('Vector Count', color=muted, fontsize=8)
            ax1.set_title('Traffic Classification Distribution', color='#f8fafc', fontsize=10, fontweight='bold', pad=10)
            ax1.tick_params(colors=muted, labelsize=8)
            ax1.spines[:].set_color('#1e293b')
            ax1.set_xlim(0, max(values)*1.18)
            plt.tight_layout()
            p1 = tempfile.NamedTemporaryFile(delete=False, suffix='_chart1.png')
            plt.savefig(p1.name, dpi=140, bbox_inches='tight', facecolor=dark_bg)
            plt.close(fig1)
            chart_paths.append(p1.name)

            # Chart 2 — Confidence distribution bar chart
            fig2, ax2 = plt.subplots(figsize=(6.5, 2.5), facecolor=dark_bg)
            ax2.set_facecolor(dark_bg)
            conf_labels = ['Mean Confidence', '95th Percentile', 'Lowest Bound']
            conf_vals   = [conf, p95, low_c]
            conf_colors = [blue_clr, '#4f46e5', '#7c3aed']
            b2 = ax2.bar(conf_labels, conf_vals, color=conf_colors, width=0.4, edgecolor='none')
            for bar, val in zip(b2, conf_vals):
                ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                         f'{val:.1f}%', ha='center', va='bottom', color='#f8fafc', fontsize=9, fontweight='bold')
            ax2.set_ylabel('Confidence (%)', color=muted, fontsize=8)
            ax2.set_title('Neural Activation Confidence Metrics', color='#f8fafc', fontsize=10, fontweight='bold', pad=10)
            ax2.tick_params(colors=muted, labelsize=8)
            ax2.spines[:].set_color('#1e293b')
            ax2.set_ylim(0, 115)
            ax2.axhline(90, color='#e11d48', linewidth=0.8, linestyle='--', alpha=0.6)
            ax2.text(2.3, 91, '90% threshold', color='#e11d48', fontsize=7, alpha=0.8)
            plt.tight_layout()
            p2 = tempfile.NamedTemporaryFile(delete=False, suffix='_chart2.png')
            plt.savefig(p2.name, dpi=140, bbox_inches='tight', facecolor=dark_bg)
            plt.close(fig2)
            chart_paths.append(p2.name)

            # Chart 3 — Threat saturation donut (pie)
            if total > 0:
                fig3, ax3 = plt.subplots(figsize=(4.5, 3.0), facecolor=dark_bg)
                ax3.set_facecolor(dark_bg)
                threat_pct = (threats / total) * 100
                wedge_sizes  = [threats, legit]
                wedge_colors = [red_clr, green_clr]
                wedge_explode= (0.04, 0)
                ax3.pie(wedge_sizes, explode=wedge_explode, colors=wedge_colors,
                        autopct='%1.1f%%', pctdistance=0.75, startangle=90,
                        wedgeprops=dict(width=0.55, edgecolor=dark_bg, linewidth=2),
                        textprops=dict(color='#f8fafc', fontsize=8))
                ax3.set_title(f'Threat Saturation: {threat_pct:.1f}%', color='#f8fafc', fontsize=10, fontweight='bold', pad=10)
                legend_patches = [mpatches.Patch(color=red_clr, label='Threats'), mpatches.Patch(color=green_clr, label='Secure')]
                ax3.legend(handles=legend_patches, loc='lower center', bbox_to_anchor=(0.5, -0.12),
                           ncol=2, frameon=False, labelcolor='#94a3b8', fontsize=8)
                plt.tight_layout()
                p3 = tempfile.NamedTemporaryFile(delete=False, suffix='_chart3.png')
                plt.savefig(p3.name, dpi=140, bbox_inches='tight', facecolor=dark_bg)
                plt.close(fig3)
                chart_paths.append(p3.name)

        except Exception as chart_err:
            logger.warning(f"[PDF] Chart generation error: {chart_err}")

        # Embed chart images into the report
        for idx, chart_path in enumerate(chart_paths):
            try:
                story.append(Image(chart_path, width=480, height=180 if idx < 2 else 230))
                story.append(Spacer(1, 14))
            except Exception:
                pass

        # Digital footer
        story.append(Paragraph("4. Chain of Custody & Execution Hash", section_header))
        story.append(Paragraph(
            "Data parameters rendered in this authorized report represent an absolute temporal snapshot of the "
            "network's condition at the time of capture. All metrics are derived from 100% of the ingested dataset "
            "processed through the quantized DNN-BiLSTM neural matrix.",
            body_text
        ))
        story.append(Spacer(1, 60))
        story.append(Paragraph("<b>END OF SECURE TRANSMISSION</b>", ParagraphStyle('End', alignment=1, textColor=colors.HexColor("#ef4444"), fontSize=12)))

        doc.build(story)

        # Clean up temp chart files
        for p in chart_paths:
            try: os.unlink(p)
            except: pass

        return pdf_path

In [ ]:
def verify_credentials(user, pwd):
    time.sleep(0.6) # Anti-brute-force simulation
    if user == "admin" and pwd == "password":
        return gr.update(visible=False), gr.update(visible=True), ""
    return gr.update(visible=True), gr.update(visible=False), "[ACCESS DENIED] INVALID CREDENTIAL HASH"

def system_lock():
    return gr.update(visible=True), gr.update(visible=False), "", "", ""

with gr.Blocks(css=enterprise_css, theme=enterprise_theme, title="IDS Global Operations Center") as master_dash:

    # -------------------------------------------------------------------------
    # System Operational Memory States
    # -------------------------------------------------------------------------
    state_model = gr.State(None)
    state_raw_data = gr.State(None)
    state_predictions = gr.State(None)
    state_probabilities = gr.State(None)
    state_confidences = gr.State(None)
    state_inference_success = gr.State(False)

    # -------------------------------------------------------------------------
    # Authentication View
    # -------------------------------------------------------------------------
    with gr.Column(visible=True, elem_classes=["auth-wrapper"]) as view_auth:
        with gr.Column(elem_classes=["auth-panel"]):
            gr.Markdown("""
            <div style="text-align: center; margin-bottom: 40px;">
                <div style="font-family: 'JetBrains Mono', monospace; font-size: 11px; color: #38bdf8; letter-spacing: 4px; font-weight: 500; margin-bottom: 12px; text-transform: uppercase;">CLASSIFIED SYSTEM</div>
                <h1 style="font-size: 28px !important; font-weight: 800 !important; letter-spacing: -0.5px !important; margin: 0 0 8px 0 !important; background: linear-gradient(135deg, #f8fafc, #94a3b8) !important; -webkit-background-clip: text !important; -webkit-text-fill-color: transparent !important;">IDS COMMAND CENTER</h1>
                <p style="color: #475569; font-size: 13px; margin: 0; font-family: 'JetBrains Mono', monospace; letter-spacing: 1px;">NEURAL EVALUATION UNIT // v9.4.2</p>
            </div>
            <div style="height: 1px; background: linear-gradient(90deg, transparent, rgba(56,189,248,0.3), transparent); margin-bottom: 36px;"></div>
            """)

            auth_user = gr.Textbox(label="OPERATOR IDENTIFIER (admin)", placeholder="Enter access credentials...")
            auth_pass = gr.Textbox(label="CRYPTOGRAPHIC TOKEN (password)", placeholder="••••••••••••", type="password")

            auth_msg = gr.Markdown("", elem_classes=["terminal-output"])
            gr.Markdown("<br>")
            btn_login = gr.Button("⚡  INITIALIZE SECURE HANDSHAKE", elem_classes=["primary-btn"])
            gr.Markdown("""
            <div style="text-align: center; margin-top: 24px;">
                <p style="color: #1e293b; font-size: 11px; font-family: 'JetBrains Mono', monospace; letter-spacing: 1px;">BIOMETRIC OVERRIDE NOT AVAILABLE · CREDENTIAL AUTH ONLY</p>
            </div>
            """)

    # -------------------------------------------------------------------------
    # Main Dashboard View
    # -------------------------------------------------------------------------
    with gr.Column(visible=False) as view_dash:

        # Header Protocol
        gr.Markdown("""
        <div class="dashboard-header">
            <div style="display: flex; align-items: center; justify-content: space-between; flex-wrap: wrap; gap: 16px;">
                <div style="display: flex; align-items: center; gap: 20px;">
                    <div>
                        <div style="font-family: 'JetBrains Mono', monospace; font-size: 10px; color: #38bdf8; letter-spacing: 4px; font-weight: 500; margin-bottom: 4px;">CLASSIFIED // TOP SECRET</div>
                        <h1 style="font-size: 26px !important; font-weight: 800 !important; margin: 0 !important; background: linear-gradient(135deg, #f8fafc 0%, #94a3b8 100%) !important; -webkit-background-clip: text !important; -webkit-text-fill-color: transparent !important; letter-spacing: -1px !important;">IDS DEEP ANALYTICS CONSOLE</h1>
                        <div style="font-family: 'JetBrains Mono', monospace; font-size: 11px; color: #475569; margin-top: 2px; letter-spacing: 1px;">DNN-BiLSTM NEURAL EVALUATION UNIT // V.9.4.2 ONLINE</div>
                    </div>
                </div>
                <div style="display: flex; align-items: center; gap: 20px;">
                    <div style="display: flex; align-items: center; gap: 8px; background: rgba(5,150,105,0.1); border: 1px solid rgba(5,150,105,0.3); border-radius: 100px; padding: 8px 16px;">
                        <div style="width: 8px; height: 8px; background: #10b981; border-radius: 50%; box-shadow: 0 0 8px #10b981; animation: pulse 2s infinite;"></div>
                        <span style="font-family: 'JetBrains Mono', monospace; font-size: 11px; color: #10b981; font-weight: 600; letter-spacing: 1px;">SYSTEMS ONLINE</span>
                    </div>
                </div>
            </div>
        </div>
        """)
        with gr.Row():
            with gr.Column(scale=12):
                gr.Markdown("")
            with gr.Column(scale=1, min_width=200):
                btn_logout = gr.Button("⏻  TERMINATE SESSION", elem_classes=["danger-btn"])

        gr.Markdown("<div style='height: 1px; background: linear-gradient(90deg, rgba(59,130,246,0.4), rgba(99,102,241,0.4), transparent); margin: 8px 0 24px 0;'></div>")

        # Deep Tabs Interface
        with gr.Tabs(elem_classes=["tabs"]):

            # =========================================================================
            # TAB 1: ENTERPRISE BATCH SCANNER
            # =========================================================================
            with gr.TabItem("ENTERPRISE BATCH SCANNER", id="tab_pipeline"):
                gr.Markdown("<p style='color:#64748b; font-size:14px;'>Upload a raw headerless CSV traffic log. The system will automatically map Index 0 and Index 3 to the neural execution graph.</p>")

                with gr.Row():
                    # Operations Card
                    with gr.Column(elem_classes=["cyber-card"], scale=1):
                        gr.Markdown("### 📂 Data Ingestion & Execution")
                        upl_csv = gr.File(label="Target Vectors Binary (.csv)")
                        btn_infer = gr.Button("⚡ EXECUTE NEURAL INFERENCE", elem_classes=["primary-btn"], size="lg", variant="primary")

                        gr.Markdown("### 📋 System Analysis Readout")
                        visual_summary = gr.HTML(label="System Analysis Readout")
                        state_summary_text = gr.State(value="")
                        state_raw_data_internal = gr.State(None)

                    # Projections Top Card
                    with gr.Column(elem_classes=["cyber-card"], scale=2):
                        gr.Markdown("### 🔍 High-Fidelity Threat Topography Projections")
                        with gr.Row():
                            with gr.Column():
                                gr.Markdown("#### System Risk Assessment\n*Aggregate gauge evaluating overall compromised node saturation across the ingest dataset.*")
                                viz_gauge = gr.Plot(show_label=False)
                            with gr.Column():
                                gr.Markdown("#### Target Hierarchy\n*Volumetric segmentation parsing discrete threat instances down by confidence intervals.*")
                                viz_sunburst = gr.Plot(show_label=False)

                # Extra Projections
                with gr.Row():
                    with gr.Column(elem_classes=["cyber-card"], scale=2):
                        gr.Markdown("#### Decision Boundary Map\n*Displays localized spatial boundaries separating anomalies (red) from legitimate nodes (green).*")
                        viz_scatter = gr.Plot(show_label=False)
                    with gr.Column(elem_classes=["cyber-card"], scale=1):
                        gr.Markdown("#### Volumetric Hex Matrix\n*Highlights depth convergence where malicious vectors overlap intensely in hyperspace.*")
                        viz_hex = gr.Plot(show_label=False)

                with gr.Row():
                    with gr.Column(elem_classes=["cyber-card"], scale=1):
                        gr.Markdown("#### Spatial Density Contours\n*Continuous KDE probability distributions illustrating regions of mathematical certainty.*")
                        viz_kde = gr.Plot(show_label=False)
                    with gr.Column(elem_classes=["cyber-card"], scale=1):
                        gr.Markdown("#### Distribution Variance\n*Analyzes neural activation spread and dispersion patterns across extracted indices.*")
                        viz_violin = gr.Plot(show_label=False)

                with gr.Row():
                    with gr.Column(elem_classes=["cyber-card"], scale=1):
                        gr.Markdown("#### 3D Topology Surface\n*Interactive three-dimensional projection tracking confidence depth against Alpha and Beta coordinates.*")
                        viz_3d = gr.Plot(show_label=False)
                    with gr.Column(elem_classes=["cyber-card"], scale=1):
                        gr.Markdown("#### Class Profile Radar\n*Comparative statistical blueprint contrasting standard metrics between clean and infected packets.*")
                        viz_radar = gr.Plot(show_label=False)

                with gr.Row():
                    with gr.Column(elem_classes=["cyber-card"], scale=1):
                        gr.Markdown("#### Softmax Distribution\n*Displays the raw probability distribution output by the final DNN-BiLSTM softmax layer.*")
                        viz_hist = gr.Plot(show_label=False)
                    with gr.Column(elem_classes=["cyber-card"], scale=1):
                        gr.Markdown("#### Feature Isolation\n*Visualizes localized quartiles and extreme outliers segregated strictly by anomaly class.*")
                        viz_box = gr.Plot(show_label=False)

                with gr.Row():
                    with gr.Column(elem_classes=["cyber-card"], scale=1):
                        gr.Markdown("#### Faceted Density Heatmaps\n*Parallel visual comparison charting exact grid saturation limits across both indices.*")
                        viz_dheat = gr.Plot(show_label=False)
                    with gr.Column(elem_classes=["cyber-card"], scale=1):
                        gr.Markdown("#### ECDF Profiling\n*Evaluates the cumulative probability accumulation to assess model confidence stabilization.*")
                        viz_ecdf = gr.Plot(show_label=False)

                with gr.Row():
                    with gr.Column(elem_classes=["cyber-card"]):
                        gr.Markdown("#### Thread Matrix Telemetry\n*High-dimensional parallel tracing tracking independent node behavior across calculated parameters.*")
                        viz_parallel = gr.Plot(show_label=False)

            # =========================================================================
            # TAB 2: ENCRYPTED ARCHIVE
            # =========================================================================
            with gr.TabItem("ENCRYPTED ARCHIVE", id="tab_export"):
                with gr.Column(elem_classes=["cyber-card"]):
                    gr.Markdown("### PDF Generation Protocol")
                    gr.Markdown("Compress analysis readout and telemetry into a formalized export structure.")
                    gr.Markdown("<br>")
                    with gr.Row():
                        with gr.Column():
                            btn_export = gr.Button("COMPILE STATIC REPORT", elem_classes=["primary-btn"])
                        with gr.Column():
                            file_export = gr.File(label="Report ready for exfiltration.", interactive=False)

    # -------------------------------------------------------------------------
    # Event Handlers & Data Binding
    # -------------------------------------------------------------------------

    # 1. Access Protocol
    btn_login.click(verify_credentials, [auth_user, auth_pass], [view_auth, view_dash, auth_msg])
    auth_pass.submit(verify_credentials, [auth_user, auth_pass], [view_auth, view_dash, auth_msg])
    btn_logout.click(system_lock, outputs=[view_auth, view_dash, auth_msg, auth_user, auth_pass])

    # 2. Ingestion Handling - Logic moved into orchestrate_systems for stability
    # (Removed upl_csv.upload listener to prevent asyncio event loop clashes in Colab)

    # 3. Execution Pipeline
    # Phase 1 — fast plots only (gauge, sunburst, scatter, violin, hist)
    def orchestrate_systems_fast(file_obj):
        if file_obj is None:
            empty_fig = go.Figure().update_layout(title="No Data", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
            return (None, None, None, None, False, "<p style='color:#f87171'>⚠️ No file uploaded.</p>", "",
                    empty_fig, empty_fig, empty_fig, empty_fig, empty_fig)

        data, msg = engine.process_data(file_obj.name)
        if data is None or len(data) == 0:
            empty_fig = go.Figure().update_layout(title="No Data", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
            return (None, None, None, None, False, f"<p style='color:#f87171'>⚠️ {msg}</p>", "",
                    empty_fig, empty_fig, empty_fig, empty_fig, empty_fig)

        res = engine.execute_inference(data)
        if not res['success']:
            empty_fig = go.Figure().update_layout(title="Inference Error", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
            return (None, None, None, None, False, f"<p style='color:#f87171'>[FATAL ERROR]</p>", "",
                    empty_fig, empty_fig, empty_fig, empty_fig, empty_fig)

        preds = res['predictions']
        probs = res['probabilities']
        conf  = res['confidence'] * 100

        total_vectors = len(preds)
        n_mal = int(np.sum(preds == 1))
        n_leg = total_vectors - n_mal
        # Use metrics from engine if available
        exec_time = res.get('metrics', {}).get('exec_time', 1.0)
        throughput = int(total_vectors / max(exec_time, 0.0001))

        summary_text = (
            "SYSTEM ANALYSIS COMPLETE\n"
            "-----------------------------------------\n"
            f"Aggregate Vectors Processed : {total_vectors:,}\n"
            f"Legitimate Nodes (Class 0)  : {n_leg:,} ({(n_leg/total_vectors)*100:.1f}%)\n"
            f"Threat Nodes (Class 1)      : {n_mal:,} ({(n_mal/total_vectors)*100:.1f}%)\n"
            f"Mean Activation Confidence  : {np.mean(conf):.2f}%\n"
            f"95th Percentile Confidence  : {np.percentile(conf, 95):.2f}%\n"
            f"Lowest Confidence Bound     : {np.min(conf):.2f}%\n"
            f"Computational Throughput    : {throughput:,} var/sec"
        )

        html_readout = f"""
        <div style='display: grid; grid-template-columns: repeat(2, 1fr); gap: 15px; margin-top: 10px;'>
            <div style='background: rgba(30, 41, 59, 0.7); padding: 15px; border-radius: 12px; border: 1px solid rgba(255,255,255,0.05); text-align: center;'>
                <p style='margin:0; font-size:11px; color:#94a3b8; font-weight:600; text-transform:uppercase; letter-spacing:0.5px;'>Processed Vectors</p>
                <h3 style='margin:5px 0 0 0; color:#38bdf8; font-size:22px;'>{total_vectors:,}</h3>
            </div>
            <div style='background: rgba(30, 41, 59, 0.7); padding: 15px; border-radius: 12px; border: 1px solid rgba(5, 150, 105, 0.2); text-align: center;'>
                <p style='margin:0; font-size:11px; color:#94a3b8; font-weight:600; text-transform:uppercase; letter-spacing:0.5px;'>Mean Confidence</p>
                <h3 style='margin:5px 0 0 0; color:#10b981; font-size:22px;'>{np.mean(conf):.1f}%</h3>
            </div>
            <div style='background: rgba(30, 41, 59, 0.7); padding: 15px; border-radius: 12px; border: 1px solid rgba(225, 29, 72, 0.2); text-align: center;'>
                <p style='margin:0; font-size:11px; color:#94a3b8; font-weight:600; text-transform:uppercase; letter-spacing:0.5px;'>Threat Nodes</p>
                <h3 style='margin:5px 0 0 0; color:#e11d48; font-size:22px;'>{n_mal:,} <span style='font-size:12px; color:#94a3b8;'>({(n_mal/total_vectors)*100:.1f}%)</span></h3>
            </div>
            <div style='background: rgba(30, 41, 59, 0.7); padding: 15px; border-radius: 12px; border: 1px solid rgba(79, 70, 229, 0.2); text-align: center;'>
                <p style='margin:0; font-size:11px; color:#94a3b8; font-weight:600; text-transform:uppercase; letter-spacing:0.5px;'>Throughput</p>
                <h3 style='margin:5px 0 0 0; color:#818cf8; font-size:22px;'>{throughput:,} <span style='font-size:12px; color:#94a3b8;'>v/s</span></h3>
            </div>
        </div>
        """

        # Sample once for all plots — reuse the same sample
        df = viz_engine.construct_dataframe(data, preds, probs, res['confidence'])
        plot_df = df.sample(n=min(5000, len(df)), random_state=42).sort_index()
        color_map = viz_engine.get_color_map()
        layout_base = viz_engine.get_layout_template()

        # 1. Gauge (uses full df for accuracy)
        threat_percent = (len(df[df['Classification'] == 'ANOMALY_THREAT']) / len(df)) * 100
        fig_gauge = go.Figure(go.Indicator(
            mode="gauge+number+delta", value=threat_percent,
            delta={'reference': 50, 'increasing': {'color': '#e11d48'}, 'decreasing': {'color': '#059669'}},
            title={'text': "System Compromise Saturation", 'font': {'size': 20}},
            number={'suffix': "%", 'font': {'size': 40}},
            gauge={
                'axis': {'range': [None, 100], 'tickwidth': 1, 'tickcolor': "#94a3b8"},
                'bar': {'color': "#3b82f6", 'thickness': 0.15},
                'bgcolor': "rgba(30, 41, 59, 0.5)",
                'borderwidth': 1, 'bordercolor': "#334155",
                'steps': [
                    {'range': [0, 20], 'color': "rgba(5, 150, 105, 0.4)"},
                    {'range': [20, 60], 'color': "rgba(217, 119, 6, 0.4)"},
                    {'range': [60, 100], 'color': "rgba(225, 29, 72, 0.4)"}],
                'threshold': {'line': {'color': "#f8fafc", 'width': 3}, 'thickness': 0.8, 'value': threat_percent}
            }
        ))
        fig_gauge.update_layout(**layout_base)

        # 2. Sunburst
        try:
            hierarchy = plot_df.copy()
            hierarchy['Conf_Level'] = pd.cut(hierarchy['Confidence_Score'], bins=[0, 75, 90, 100], labels=['Low', 'Medium', 'High'])
            hierarchy_data = hierarchy.groupby(['Classification', 'Conf_Level'], observed=True).size().reset_index(name='Volume')
            fig_sunburst = px.sunburst(hierarchy_data, path=['Classification', 'Conf_Level'], values='Volume',
                                   color='Classification', color_discrete_map=color_map, title="Hierarchical Traffic Segmentation")
            fig_sunburst.update_layout(**layout_base)
        except Exception:
            fig_sunburst = go.Figure().update_layout(title="Sunburst Unavailable", **layout_base)

        # 3. Scatter
        try:
            fig_scatter = px.scatter(plot_df, x="Alpha Spatial Map (Index 0)", y="Beta Spatial Map (Index 3)",
                                 color="Classification", color_discrete_map=color_map,
                                 title="Bivariate Topography (Alpha vs Beta)", opacity=0.7,
                                 marginal_x="box", marginal_y="box", render_mode="webgl")
            fig_scatter.update_layout(**layout_base)
        except Exception:
            fig_scatter = go.Figure().update_layout(title="Scatter Unavailable", **layout_base)

        # 4. Violin
        try:
            fig_violin = px.violin(plot_df, y="Confidence_Score", x="Classification", color="Classification",
                               box=True, points="outliers", color_discrete_map=color_map,
                               title="Neural Network Activation Variance")
            fig_violin.update_layout(**layout_base)
        except Exception:
            fig_violin = go.Figure().update_layout(title="Violin Unavailable", **layout_base)

        # 5. Histogram
        try:
            fig_hist = px.histogram(plot_df, x="Anomaly_Probability", color="Classification", marginal="rug",
                                color_discrete_map=color_map, opacity=0.75, nbins=100,
                                title="Raw Softmax Probability Histogram")
            fig_hist.update_layout(**layout_base)
        except Exception:
            fig_hist = go.Figure().update_layout(title="Histogram Unavailable", **layout_base)

        return (preds, probs, conf, data, True, html_readout, summary_text,
                fig_gauge, fig_sunburst, fig_scatter, fig_violin, fig_hist)


    # Phase 2 — heavy plots (3d, hex, kde, parallel, box, dheat, ecdf, radar)
    def orchestrate_systems_slow(preds, probs, conf_pct, data, success):
        empty_fig = go.Figure().update_layout(title="No Data", paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)')
        if not success or data is None:
            return (empty_fig,) * 8

        # Correct confidence scale back to 0-1 for construction
        conf_raw = conf_pct / 100
        df = viz_engine.construct_dataframe(data, preds, probs, conf_raw)
        plot_df = df.sample(n=min(5000, len(df)), random_state=42).sort_index()
        color_map = viz_engine.get_color_map()
        layout_base = viz_engine.get_layout_template()

        # 3D
        try:
            fig_3d = px.scatter_3d(plot_df, x='Alpha Spatial Map (Index 0)', y='Beta Spatial Map (Index 3)',
                                   z='Confidence_Score', color='Classification', color_discrete_map=color_map,
                                   opacity=0.7, size_max=10, title="3D Dimensional Threat Topology", hover_data=['Node_ID'])
            fig_3d.update_layout(**layout_base)
            fig_3d.update_layout(scene=dict(
                xaxis_title='Feature Alpha', yaxis_title='Feature Beta', zaxis_title='Confidence %',
                xaxis=dict(gridcolor='#334155', backgroundcolor='rgba(0,0,0,0)'),
                yaxis=dict(gridcolor='#334155', backgroundcolor='rgba(0,0,0,0)'),
                zaxis=dict(gridcolor='#334155', backgroundcolor='rgba(0,0,0,0)')))
        except Exception:
            fig_3d = empty_fig

        # Hex
        try:
            fig_hex = px.density_heatmap(plot_df, x='Alpha Spatial Map (Index 0)', y='Beta Spatial Map (Index 3)',
                                         color_continuous_scale=['#1e293b', '#3b82f6', '#e11d48'],
                                         nbinsx=30, nbinsy=30, title="Volumetric Convergence Matrix")
            fig_hex.update_layout(**layout_base)
        except Exception:
            fig_hex = empty_fig

        # KDE
        try:
            fig_kde = px.density_contour(plot_df, x="Alpha Spatial Map (Index 0)", y="Beta Spatial Map (Index 3)",
                                         color="Classification", color_discrete_map=color_map,
                                         title="Topographical Probability Contours",
                                         marginal_x="histogram", marginal_y="histogram")
            fig_kde.update_traces(contours_coloring="fill", opacity=0.4, selector=dict(type="histogram2dcontour"))
            fig_kde.update_layout(**layout_base)
        except Exception:
            fig_kde = empty_fig

        # Parallel
        try:
            plot_df_pc = plot_df.copy()
            plot_df_pc['Label_ID'] = [1 if x == 'ANOMALY_THREAT' else 0 for x in plot_df_pc['Classification']]
            fig_parallel = px.parallel_coordinates(plot_df_pc, color="Label_ID",
                                                   dimensions=['Alpha Spatial Map (Index 0)', 'Beta Spatial Map (Index 3)',
                                                               'Confidence_Score', 'Anomaly_Probability'],
                                                   color_continuous_scale=[[0, '#059669'], [1, '#e11d48']],
                                                   title="Parallel Feature Dimension Matrix")
            fig_parallel.update_layout(**layout_base)
        except Exception:
            fig_parallel = empty_fig

        # Box
        try:
            fig_box = make_subplots(rows=1, cols=2, subplot_titles=("Alpha Spatial Distribution", "Beta Spatial Distribution"))
            for trace in px.box(plot_df, x="Classification", y="Alpha Spatial Map (Index 0)",
                                color="Classification", color_discrete_map=color_map).data:
                fig_box.add_trace(trace, row=1, col=1)
            for trace in px.box(plot_df, x="Classification", y="Beta Spatial Map (Index 3)",
                                color="Classification", color_discrete_map=color_map).data:
                fig_box.add_trace(trace, row=1, col=2)
            fig_box.update_layout(**layout_base)
            fig_box.update_layout(title_text="Feature Isolation Boxplots", showlegend=False)
        except Exception:
            fig_box = empty_fig

        # Density Heatmap faceted
        try:
            fig_dheat = px.density_heatmap(plot_df, x="Alpha Spatial Map (Index 0)", y="Beta Spatial Map (Index 3)",
                                           facet_col="Classification", marginal_x="rug", marginal_y="histogram",
                                           title="Faceted Density Heatmaps")
            fig_dheat.update_layout(**layout_base)
        except Exception:
            fig_dheat = empty_fig

        # ECDF
        try:
            fig_ecdf = px.ecdf(plot_df, x="Confidence_Score", color="Classification",
                               color_discrete_map=color_map, title="Empirical Cumulative Distribution of Confidence")
            fig_ecdf.update_layout(**layout_base)
        except Exception:
            fig_ecdf = empty_fig

        # Radar
        try:
            metrics_list = ['Mean Alpha', 'Mean Beta', 'Std Alpha', 'Std Beta', 'Mean Confidence']
            def get_metrics_vals(cls):
                sub = plot_df[plot_df['Classification'] == cls]
                return [sub['Alpha Spatial Map (Index 0)'].mean(), sub['Beta Spatial Map (Index 3)'].mean(),
                sub['Alpha Spatial Map (Index 0)'].std(), sub['Beta Spatial Map (Index 3)'].std(),
                sub['Confidence_Score'].mean() / 100]
            fig_radar = go.Figure()
            fig_radar.add_trace(go.Scatterpolar(r=get_metrics_vals('SECURE_NODE'), theta=metrics_list, fill='toself',
                                                name='SECURE_NODE', line_color='#059669'))
            fig_radar.add_trace(go.Scatterpolar(r=get_metrics_vals('ANOMALY_THREAT'), theta=metrics_list, fill='toself',
                                                name='ANOMALY_THREAT', line_color='#e11d48'))
            fig_radar.update_layout(**layout_base)
            fig_radar.update_layout(polar=dict(radialaxis=dict(visible=True, range=[-3, 3])), title="Class Profile Radar")
        except Exception:
            fig_radar = empty_fig

        return fig_3d, fig_hex, fig_kde, fig_parallel, fig_box, fig_dheat, fig_ecdf, fig_radar

    fast_event = btn_infer.click(
        orchestrate_systems_fast,
        inputs=[upl_csv],
        outputs=[
            state_predictions, state_probabilities, state_confidences,
            state_raw_data_internal,
            state_inference_success, visual_summary, state_summary_text,
            viz_gauge, viz_sunburst, viz_scatter, viz_violin, viz_hist
        ],
        show_progress="full",
        queue=False
    )

    fast_event.then(
        orchestrate_systems_slow,
        inputs=[state_predictions, state_probabilities, state_confidences,
                state_raw_data_internal, state_inference_success],
        outputs=[viz_3d, viz_hex, viz_kde, viz_parallel, viz_box, viz_dheat, viz_ecdf, viz_radar],
        queue=False
    )

    # 4. Intelligence Optics Chain - REMOVED (now handled atomically in stages)

    # 5. Archive Compilation
    def handle_pdf_export(summary_text, success):
        # Pass the plain string to the PDF Engine, since it handles splitting internally
        return EnterprisePDFEngine.build_report(summary_text, success)

    btn_export.click(
        handle_pdf_export,
        inputs=[state_summary_text, state_inference_success],
        outputs=[file_export]
    )

In [ ]:
if __name__ == "__main__":
    print("Initializing Advanced Dashboard Server...")
    master_dash.queue(max_size=5).launch(debug=False, share=True)

Initializing Advanced Dashboard Server...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://55b9261e87b3d925ab.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
